# Task 1 CRNN Training on Colab

This notebook runs the `task1_crnn` training and evaluation pipeline on Google Colab.

Important:
- Use a **fresh** `wikiart.zip` in Google Drive.
- Do not use the corrupted local archive from this Mac.
- GPU runtime is recommended: `Runtime -> Change runtime type -> T4 GPU`.

## Expected Drive Layout

```text
MyDrive/
  GSoC/
    ArtGAN/
      colab/
      task1_crnn/
      WikiArt Dataset/
      TASK1_CONV_RECURRENT.md
    wikiart.zip
```

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path

MYDRIVE = Path('/content/drive/MyDrive')

# This notebook prefers your current Drive tree: MyDrive/GSoC/ArtGAN
# but falls back to the older MyDrive/ArtGAN layout if needed.
REPO_CANDIDATES = [
    MYDRIVE / 'GSoC' / 'ArtGAN',
    MYDRIVE / 'ArtGAN',
]
DATA_ZIP_CANDIDATES = [
    MYDRIVE / 'GSoC' / 'wikiart.zip',
    MYDRIVE / 'wikiart.zip',
    MYDRIVE / 'GSoC' / 'ArtGAN' / 'wikiart.zip',
]

DRIVE_REPO_DIR = next((path for path in REPO_CANDIDATES if path.exists()), REPO_CANDIDATES[0])
DRIVE_DATA_ZIP = next((path for path in DATA_ZIP_CANDIDATES if path.exists()), DATA_ZIP_CANDIDATES[0])
PROJECT_ROOT = DRIVE_REPO_DIR.parent

WORKDIR = Path('/content/ArtGAN')
LOCAL_DATA_ZIP = Path('/content/wikiart.zip')
OUTPUTS_DIR = PROJECT_ROOT / 'ArtGAN_outputs' / 'task1_crnn'

print('REPO_CANDIDATES =', REPO_CANDIDATES)
print('DATA_ZIP_CANDIDATES =', DATA_ZIP_CANDIDATES)
print('DRIVE_REPO_DIR =', DRIVE_REPO_DIR)
print('DRIVE_DATA_ZIP =', DRIVE_DATA_ZIP)
print('PROJECT_ROOT =', PROJECT_ROOT)
print('WORKDIR =', WORKDIR)
print('LOCAL_DATA_ZIP =', LOCAL_DATA_ZIP)
print('OUTPUTS_DIR =', OUTPUTS_DIR)

In [ ]:
import os
import shutil

if not DRIVE_REPO_DIR.exists():
    raise FileNotFoundError(
        'Missing repo folder. Checked: ' + ', '.join(str(path) for path in REPO_CANDIDATES)
    )
if not DRIVE_DATA_ZIP.exists():
    raise FileNotFoundError(
        'Missing dataset zip. Checked: ' + ', '.join(str(path) for path in DATA_ZIP_CANDIDATES)
    )

if WORKDIR.exists():
    shutil.rmtree(WORKDIR)
shutil.copytree(DRIVE_REPO_DIR, WORKDIR, dirs_exist_ok=True)
shutil.copy2(DRIVE_DATA_ZIP, LOCAL_DATA_ZIP)
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(WORKDIR)

print('Copied repo to', WORKDIR)
print('Copied dataset zip to', LOCAL_DATA_ZIP)
print('Outputs will be saved under', OUTPUTS_DIR)
print('Current working directory:', Path.cwd())

In [ ]:
%pip install -q scikit-learn pillow tqdm

In [ ]:
!python -m task1_crnn.audit --dataset-dir "WikiArt Dataset" --output-json outputs/task1_crnn/audit.json

## Smoke Test

Run this first to verify the environment before launching the full training job.

For Colab stability with the large `wikiart.zip`, this notebook uses `num_workers=0` so image loading stays in the main process.

In [ ]:
!python -m task1_crnn.train \
  --dataset-dir "WikiArt Dataset" \
  --archive-path "/content/wikiart.zip" \
  --output-dir outputs/task1_crnn_smoke \
  --epochs 1 \
  --batch-size 4 \
  --num-workers 0 \
  --limit-train-batches 10 \
  --limit-val-batches 10 \
  --disable-progress \
  --image-size 224 \
  --crop-size 224

## Full Training

If your Colab session stalls partway through an epoch, keep `num_workers=0` and start with batch size `8`.

If that still stops, lower batch size to `4` and verify that `wikiart.zip` is a fresh download.

In [ ]:
!python -m task1_crnn.train \
  --dataset-dir "WikiArt Dataset" \
  --archive-path "/content/wikiart.zip" \
  --output-dir outputs/task1_crnn \
  --epochs 12 \
  --batch-size 8 \
  --num-workers 0 \
  --disable-progress

## Evaluation and Outliers

In [ ]:
!python -m task1_crnn.evaluate \
  --checkpoint outputs/task1_crnn/best.pt \
  --dataset-dir "WikiArt Dataset" \
  --archive-path "/content/wikiart.zip" \
  --output-dir outputs/task1_crnn/eval \
  --batch-size 8 \
  --num-workers 0 \
  --disable-progress \
  --top-outliers 25

In [ ]:
import shutil

target = OUTPUTS_DIR
target.mkdir(parents=True, exist_ok=True)
if (target / 'task1_crnn').exists():
    shutil.rmtree(target / 'task1_crnn')
shutil.copytree(WORKDIR / 'outputs' / 'task1_crnn', target / 'task1_crnn', dirs_exist_ok=True)
print('Saved outputs to', target / 'task1_crnn')

## Optional: Clone from GitHub Instead of Copying the Folder from Drive

If your code is on GitHub, you can replace the copy step with:

```bash
!git clone <YOUR_REPO_URL> /content/ArtGAN
```

Then keep using the same training and evaluation cells.